# Build Per-Trait Sliced-Dual FAISS Indexes

Builds **5 separate FAISS indexes**, one per personality trait.
Each training vector uses only the facets relevant to that trait:

```
vector[trait] = normalise( α·embed(raw) + (1−α)·embed(slice_profile_for_trait(profile, trait)) )
```

This is the symmetric counterpart to the asymmetric sliced-dual comparison notebook.
At retrieval time, both the query and the index use the same 6-facet slice → no cross-trait noise.

**Efficiency:** raw posts are encoded once and shared across all 5 traits.
Total encoding calls: `N_train + 5×N_train` instead of `5×2×N_train`.

**Prerequisites:**
- `data/split/essays/train.csv`
- `data/profile_db/essays/profile_store.jsonl` (built by profiler)

**Output:** `data/vector_db/essays_sliced_dual/{trait_code}/vectors.faiss` + `vectors_meta.jsonl`

## 1. Config

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = str(Path(os.getcwd()).parent.parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

TRAIN_CSV          = os.path.join(PROJECT_ROOT, "data", "split", "essays", "train.csv")
PROFILE_STORE_PATH = os.path.join(PROJECT_ROOT, "data", "profile_db", "essays", "profile_store.jsonl")
OUTPUT_BASE        = os.path.join(PROJECT_ROOT, "data", "vector_db", "essays_sliced_dual")

FINETUNED_MODEL_DIR = os.path.join(PROJECT_ROOT, "models", "sbert_essays_finetuned")
BASE_MODEL          = "nomic-ai/nomic-embed-text-v1.5"
ALPHA               = 0.5
FORCE_REBUILD       = False   # set True to overwrite existing indexes

TRAITS = {
    "cOPN": "Openness to Experience",
    "cCON": "Conscientiousness",
    "cEXT": "Extraversion",
    "cAGR": "Agreeableness",
    "cNEU": "Neuroticism",
}

print("PATH CHECK")
for label, p in [("TRAIN_CSV", TRAIN_CSV), ("PROFILE_STORE", PROFILE_STORE_PATH)]:
    print(f"  {label:18s}: {'OK' if Path(p).exists() else 'MISSING'}  {p}")
print(f"  OUTPUT_BASE       : {OUTPUT_BASE}")
print(f"  ALPHA             : {ALPHA}")
print(f"  FORCE_REBUILD     : {FORCE_REBUILD}")

## 2. Imports

In [ ]:
import json
import time
import numpy as np
import pandas as pd

import rag.embedder as _embedder
_embedder.FINETUNED_MODEL_DIR = FINETUNED_MODEL_DIR
_embedder.ALPHA               = ALPHA

from rag.embedder         import _embed_single, get_embedding
from rag.profiler.store   import ProfileStore
from rag.profiler.prompts import slice_profile_for_trait
from rag.faiss_index      import FAISSIndex

print("Imports OK")

## 3. Load train data and profiles

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
print(f"Train: {len(train_df)} essays")

store = ProfileStore(PROFILE_STORE_PATH)
store.load()
print(f"Profiles: {len(store)}")

valid_count = sum(
    1 for i in range(len(train_df))
    if store.has(f"user_{i}") and store.get(f"user_{i}").get("valid")
)
print(f"  Valid : {valid_count} / {len(train_df)} ({valid_count/len(train_df)*100:.1f}%)")
print(f"  Fallback (raw text) : {len(train_df) - valid_count}")

## 4. Encode raw posts once (shared across all traits)

In [ ]:
N = len(train_df)
raw_texts = [str(row["text"]) for _, row in train_df.iterrows()]

print(f"Encoding {N} raw posts ...")
t0 = time.time()
raw_embs = np.array(
    [_embed_single(t) for t in raw_texts],
    dtype="float32",
)
print(f"Done in {time.time()-t0:.1f}s  shape={raw_embs.shape}")

## 5. Build one index per trait

For each trait: encode sliced profile texts, fuse with the cached raw embeddings, save FAISS index.

In [ ]:
def _fuse(raw_emb: np.ndarray, prof_emb: np.ndarray, alpha: float = ALPHA) -> np.ndarray:
    fused = alpha * raw_emb + (1.0 - alpha) * prof_emb
    norm  = np.linalg.norm(fused)
    return (fused / norm) if norm > 0 else fused


built_summary = []

for trait_code, trait_full in TRAITS.items():
    out_dir    = os.path.join(OUTPUT_BASE, trait_code)
    index_path = os.path.join(out_dir, "vectors.faiss")
    meta_path  = os.path.join(out_dir, "vectors_meta.jsonl")

    if not FORCE_REBUILD and os.path.exists(index_path) and os.path.exists(meta_path):
        print(f"[{trait_code}] Already exists — skipping. Set FORCE_REBUILD=True to overwrite.")
        built_summary.append((trait_code, "skipped", out_dir))
        continue

    print(f"\n[{trait_code}] Encoding {N} sliced profiles ({trait_full}) ...")
    t0 = time.time()

    sliced_texts = []
    fallbacks    = 0
    for i, raw_text in enumerate(raw_texts):
        entry = store.get(f"user_{i}")
        if entry and entry.get("valid"):
            t = slice_profile_for_trait(entry, trait_code)
        else:
            t = ""
        if t.strip():
            sliced_texts.append(t)
        else:
            sliced_texts.append(raw_text)   # fallback: use raw text on profile side too
            fallbacks += 1

    prof_embs = np.array(get_embedding(sliced_texts), dtype="float32")
    print(f"  Profile slices encoded in {time.time()-t0:.1f}s  (fallbacks: {fallbacks})")

    # Fuse raw + sliced profile
    fused_embs = np.array(
        [_fuse(raw_embs[i], prof_embs[i]) for i in range(N)],
        dtype="float32",
    )

    # Build metadata
    meta = []
    for i, (_, row) in enumerate(train_df.iterrows()):
        rag_labels = {}
        for tc, tf in TRAITS.items():
            v = str(row.get(tc, "")).strip().lower()
            if v in ("high", "low"):       rag_labels[tf] = v
            elif v in ("1", "1.0"):         rag_labels[tf] = "high"
            elif v in ("0", "0.0"):         rag_labels[tf] = "low"
        entry = store.get(f"user_{i}")
        meta.append({
            "user_id":      f"user_{i}",
            "posts_raw":    raw_texts[i],
            "trait_labels": rag_labels,
            "profile":      {"facets": entry.get("facets", {})} if entry else {},
        })

    # Save
    os.makedirs(out_dir, exist_ok=True)
    idx = FAISSIndex(dimension=fused_embs.shape[1])
    idx.build(fused_embs, meta)
    idx.save(index_path, meta_path)
    print(f"  Saved -> {out_dir}  ({N} vectors, dim={fused_embs.shape[1]})")
    built_summary.append((trait_code, "built", out_dir))

print("\n=== Summary ===")
for tc, status, path in built_summary:
    size_mb = sum(
        os.path.getsize(os.path.join(path, f)) / 1e6
        for f in os.listdir(path)
        if os.path.isfile(os.path.join(path, f))
    ) if os.path.exists(path) else 0
    print(f"  {tc}  [{status:7s}]  {size_mb:.1f} MB  {path}")

## 6. Sanity check — symmetric retrieval accuracy on val

For each trait: encode val essays with the matching sliced-dual query, search the per-trait index,
majority-vote the label. Requires val profiles at `data/profile_db/essays_val/`.

In [ ]:
from pathlib import Path as _Path

VAL_CSV            = os.path.join(PROJECT_ROOT, "data", "split", "essays", "val.csv")
VAL_PROFILE_STORE  = os.path.join(PROJECT_ROOT, "data", "profile_db", "essays_val", "profile_store.jsonl")
TOP_K              = 5

if not _Path(VAL_PROFILE_STORE).exists():
    print("Val profiles not found — skipping sanity check.")
    print(f"Run notebook/rag_compare/rag_retrieve_accuracy_profile.ipynb to build them.")
else:
    from rag.profiler.store import ProfileStore as _PS
    from sklearn.metrics import accuracy_score

    val_df    = pd.read_csv(VAL_CSV)
    val_store = _PS(VAL_PROFILE_STORE)
    val_store.load()
    val_profiles = {
        int(e["user_id"].split("_")[1]): e
        for e in val_store.get_all() if e.get("valid")
    }
    print(f"Val: {len(val_df)} essays, profiles: {len(val_profiles)}")

    def _norm_label(v):
        s = str(v).strip().lower()
        if s in ("1", "1.0"): return "high"
        if s in ("0", "0.0"): return "low"
        return s

    print(f"\n=== Symmetric sliced-dual retrieval accuracy (top-{TOP_K} majority vote) ===\n")
    print(f"  {'Trait':<30s} {'Acc':>8s}")
    print("  " + "-" * 40)

    for trait_code, trait_full in TRAITS.items():
        idx_path  = os.path.join(OUTPUT_BASE, trait_code, "vectors.faiss")
        meta_path = os.path.join(OUTPUT_BASE, trait_code, "vectors_meta.jsonl")
        if not os.path.exists(idx_path):
            print(f"  {trait_full:<30s} index missing — skipped")
            continue

        trait_idx = FAISSIndex(dimension=0)
        trait_idx.load(idx_path, meta_path)

        val_labels, preds = [], []
        for i, (_, row) in enumerate(val_df.iterrows()):
            gt = _norm_label(row[trait_code])
            if gt not in ("high", "low"):
                continue
            val_labels.append(gt)

            raw_text = str(row["text"])
            pe = val_profiles.get(i)
            if pe:
                sliced_text = slice_profile_for_trait(pe, trait_code)
            else:
                sliced_text = ""
            profile_text = sliced_text if sliced_text.strip() else raw_text

            raw_v  = _embed_single(raw_text)
            prof_v = np.array(get_embedding(profile_text), dtype="float32")
            q_emb  = _fuse(raw_v, prof_v)

            _, results = trait_idx.search(q_emb, TOP_K * 6)
            nbrs = [r for r in results if trait_full in r.get("trait_labels", {})][:TOP_K]
            labels = [r["trait_labels"][trait_full] for r in nbrs]
            preds.append("high" if labels.count("high") >= labels.count("low") else "low")

        acc = accuracy_score(val_labels, preds)
        print(f"  {trait_full:<30s} {acc:>8.4f}")